CELL 1 — Install ALL required packages

In [7]:
# ============================================================
# CELL 1 — INSTALL ALL REQUIRED DEPENDENCIES
# ============================================================

%pip install -q --upgrade pip

%pip install -q \
    pymysql \
    pandas \
    numpy \
    matplotlib \
    scikit-learn \
    tqdm \
    selfies \
    rdkit \
    torch \
    torch-geometric

print("Package installation complete. Proceed to Cell 2.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Package installation complete. Proceed to Cell 2.


CELL 2 — Phase 1 Implementation

In [10]:
# ============================================================
# CELL 2 — PHASE 1: DRUGBANK EXTRACTION & ROBUST 3D GRAPHS
# ============================================================

import os
import sys
import random
import warnings
import pymysql
from pymysql.cursors import DictCursor

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
from torch_geometric.data import Data

from rdkit import Chem, DataStructs
from rdkit.Chem import (
    AllChem,
    Descriptors,
    Lipinski,
    rdMolDescriptors,
    rdFingerprintGenerator
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Reproducibility & Device Configuration
# ------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Operating Device:", DEVICE)

# ------------------------------------------------------------
# 2. Local XAMPP MySQL Database Configuration
# ------------------------------------------------------------
DB_CONFIG = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "",
    "database": "drugbank",
    "charset": "utf8mb4",
    "cursorclass": DictCursor
}

MAX_MOLECULES = 600
MAX_HEAVY_ATOMS = 150
FP_DIM = 256

# ------------------------------------------------------------
# 3. Diagnostic Database Probing & Extraction
# ------------------------------------------------------------
try:
    connection = pymysql.connect(**DB_CONFIG)
    print("Connected successfully to MySQL database:", DB_CONFIG["database"])
except Exception as exc:
    raise RuntimeError(
        f"\n[FATAL] Could not connect to MySQL.\n"
        f"Ensure XAMPP MySQL is active on port 3306 and database 'drugbank' exists.\n"
        f"Original error: {exc}"
    )

with connection.cursor() as cursor:
    # Inspect distinct kind values in drug_property to avoid naming mismatches
    cursor.execute("SELECT DISTINCT kind FROM drug_property WHERE kind LIKE '%SMILES%' OR kind LIKE '%smiles%';")
    kinds_found = cursor.fetchall()
    print(f"Distinct SMILES property keys found in DB: {kinds_found}")

# Resilient query selecting calculated/experimental SMILES across naming variants
SMILES_QUERY = """
SELECT DISTINCT
    d.drug_pk,
    d.primary_drugbank_id AS drug_id,
    d.name AS drug_name,
    dp.value AS smiles
FROM drug AS d
INNER JOIN drug_property AS dp
    ON d.drug_pk = dp.drug_pk
WHERE (LOWER(dp.kind) = 'smiles' OR LOWER(dp.kind) LIKE '%%smiles%%')
  AND dp.value IS NOT NULL
  AND TRIM(dp.value) <> ''
LIMIT %s;
"""

TARGET_QUERY = """
SELECT DISTINCT
    d.drug_pk,
    d.primary_drugbank_id AS drug_id,
    d.name AS drug_name,
    dp.value AS smiles,
    i.interactant_pk,
    i.interactant_id,
    i.name AS target_name,
    i.organism AS target_organism,
    i.known_action,
    ip.polypeptide_id,
    ip.source AS polypeptide_source,
    p.amino_acid_sequence
FROM drug AS d
INNER JOIN drug_property AS dp
    ON d.drug_pk = dp.drug_pk
   AND (LOWER(dp.kind) = 'smiles' OR LOWER(dp.kind) LIKE '%%smiles%%')
INNER JOIN interactant AS i
    ON d.drug_pk = i.drug_pk
   AND LOWER(i.kind) = 'target'
INNER JOIN interactant_polypeptide AS ip
    ON i.interactant_pk = ip.interactant_pk
INNER JOIN polypeptide AS p
    ON ip.polypeptide_id = p.polypeptide_id
   AND ip.source = p.source
WHERE dp.value IS NOT NULL
  AND TRIM(dp.value) <> ''
  AND p.amino_acid_sequence IS NOT NULL
LIMIT %s;
"""

chem_df = pd.DataFrame()
target_df = pd.DataFrame()

try:
    chem_df = pd.read_sql(SMILES_QUERY, connection, params=(MAX_MOLECULES,))
    target_df = pd.read_sql(TARGET_QUERY, connection, params=(MAX_MOLECULES,))
finally:
    connection.close()

print(f"Extracted chemical records (D_chem): {len(chem_df)}")
print(f"Extracted drug-target records (D_val): {len(target_df)}")

if len(chem_df) == 0:
    raise RuntimeError(
        "[FATAL] Zero chemical records found in table 'drug_property'. "
        "Verify your kind column values from the distinct printout above."
    )

# ------------------------------------------------------------
# 4. SMILES Sanitization (Largest Organic Fragment Selection)
# ------------------------------------------------------------
def sanitize_smiles(raw_smiles):
    try:
        if not isinstance(raw_smiles, str) or not raw_smiles.strip():
            return None
        # Split salts and counter-ions (e.g., [Na+].[Cl-] or hydrochloride mixtures)
        if "." in raw_smiles:
            fragments = raw_smiles.split(".")
            raw_smiles = max(fragments, key=len)

        mol = Chem.MolFromSmiles(raw_smiles)
        if mol is None or mol.GetNumHeavyAtoms() == 0:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

chem_df["smiles"] = chem_df["smiles"].apply(sanitize_smiles)
chem_df = chem_df.dropna(subset=["smiles"]).drop_duplicates(subset=["smiles"]).reset_index(drop=True)
print(f"Sanitized single-molecule candidates: {len(chem_df)}")

# ------------------------------------------------------------
# 5. Graph Feature Extractors
# ------------------------------------------------------------
FP_GENERATOR = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_DIM)

def get_fingerprint_array(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = FP_GENERATOR.GetFingerprint(mol)
    arr = np.zeros((FP_DIM,), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def atom_features(atom):
    hybridizations = [
        Chem.rdchem.HybridizationType.SP,
        Chem.rdchem.HybridizationType.SP2,
        Chem.rdchem.HybridizationType.SP3,
    ]
    hyb_onehot = [float(atom.GetHybridization() == h) for h in hybridizations]
    hyb_onehot.append(float(atom.GetHybridization() not in hybridizations))

    chiral_tags = [
        Chem.rdchem.ChiralType.CHI_UNSPECIFIED,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CW,
        Chem.rdchem.ChiralType.CHI_TETRAHEDRAL_CCW,
    ]
    chiral_onehot = [float(atom.GetChiralTag() == tag) for tag in chiral_tags]
    chiral_onehot.append(float(atom.GetChiralTag() not in chiral_tags))

    return [
        atom.GetAtomicNum() / 100.0,
        atom.GetFormalCharge() / 5.0,
        atom.GetDegree() / 6.0,
        float(atom.GetIsAromatic()),
        *hyb_onehot,
        *chiral_onehot,
    ]

def bond_features(bond):
    bt = bond.GetBondType()
    return [
        float(bt == Chem.rdchem.BondType.SINGLE),
        float(bt == Chem.rdchem.BondType.DOUBLE),
        float(bt == Chem.rdchem.BondType.TRIPLE),
        float(bt == Chem.rdchem.BondType.AROMATIC),
        float(bond.GetIsConjugated()),
    ]

# ------------------------------------------------------------
# 6. Attributed 3D Molecular Graph Construction G = (V, E, X, R)
# ------------------------------------------------------------
def smiles_to_spatial_graph(smiles, drug_id="UNKNOWN", drug_name="UNKNOWN"):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None or mol.GetNumHeavyAtoms() == 0 or mol.GetNumHeavyAtoms() > MAX_HEAVY_ATOMS:
            return None

        coords = None

        # Tier 1: Try ETKDGv3 3D Conformation
        try:
            mol_h = Chem.AddHs(mol)
            params = AllChem.ETKDGv3()
            params.randomSeed = SEED
            params.maxAttempts = 10
            params.useRandomCoords = True
            
            if AllChem.EmbedMolecule(mol_h, params) == 0:
                mol_no_h = Chem.RemoveHs(mol_h)
                if mol_no_h.GetNumConformers() > 0:
                    conf = mol_no_h.GetConformer()
                    coords = [list(conf.GetAtomPosition(i)) for i in range(mol_no_h.GetNumAtoms())]
                    mol = mol_no_h
        except Exception:
            coords = None

        # Tier 2: Planar 2D Conformer Projected to 3D (Z=0.0) Fallback
        if coords is None:
            AllChem.Compute2DCoords(mol)
            if mol.GetNumConformers() > 0:
                conf = mol.GetConformer()
                coords = [[conf.GetAtomPosition(i).x, conf.GetAtomPosition(i).y, 0.0] for i in range(mol.GetNumAtoms())]
            else:
                coords = [[float(i), 0.0, 0.0] for i in range(mol.GetNumAtoms())]

        # Node Feature Matrix (X)
        x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)

        # 3D Cartesian Coordinates (R) centered at coordinate origin
        pos = torch.tensor(coords, dtype=torch.float32)
        pos = pos - pos.mean(dim=0, keepdim=True)

        # Chemical Bonds (E)
        src, dst, edge_attr = [], [], []
        for bond in mol.GetBonds():
            u, v = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
            feats = bond_features(bond)
            src.extend([u, v])
            dst.extend([v, u])
            edge_attr.extend([feats, feats])

        if len(src) == 0:
            edge_index = torch.empty((2, 0), dtype=torch.long)
            edge_attr_tensor = torch.empty((0, 5), dtype=torch.float32)
        else:
            edge_index = torch.tensor([src, dst], dtype=torch.long)
            edge_attr_tensor = torch.tensor(edge_attr, dtype=torch.float32)

        fp = get_fingerprint_array(smiles)
        if fp is None:
            return None

        graph = Data(x=x, pos=pos, edge_index=edge_index, edge_attr=edge_attr_tensor)
        
        # 1D tensor [256] avoiding shape issues during DataLoader collation
        graph.fp = torch.tensor(fp, dtype=torch.float32)
        
        graph.smiles = smiles
        graph.drug_id = str(drug_id)
        graph.drug_name = str(drug_name)
        return graph

    except Exception:
        return None

# ------------------------------------------------------------
# 7. Build Graph Corpus
# ------------------------------------------------------------
baseline_graphs = []
for row in tqdm(chem_df.itertuples(index=False), total=len(chem_df), desc="Constructing 3D Attributed Graphs"):
    g = smiles_to_spatial_graph(row.smiles, drug_id=row.drug_id, drug_name=row.drug_name)
    if g is not None:
        baseline_graphs.append(g)

print(f"\nPhase 1 Complete: {len(baseline_graphs)} spatial graphs successfully constructed.")

if len(baseline_graphs) == 0:
    raise RuntimeError("[FATAL] Zero molecular graphs were created from the extracted DrugBank SMILES.")

NODE_FEATURE_DIM = baseline_graphs[0].x.shape[1]
EDGE_FEATURE_DIM = baseline_graphs[0].edge_attr.shape[1]
print(f"Graph Dimensions -> Node Feature Dim (X): {NODE_FEATURE_DIM}, Bond Feature Dim (E): {EDGE_FEATURE_DIM}")

# ------------------------------------------------------------
# 8. Train / Validation Split for Phase-2 Continual Learning
# ------------------------------------------------------------
random.shuffle(baseline_graphs)
split_idx = int(0.85 * len(baseline_graphs))

train_graphs = baseline_graphs[:split_idx]
validation_graphs = baseline_graphs[split_idx:]

print(f"\nDataset Partitioning:")
print(f" - Baseline Training Graphs   (D_chem pretraining): {len(train_graphs)}")
print(f" - Baseline Validation Graphs (D_val retention)  : {len(validation_graphs)}")

display(chem_df.head(3))

Operating Device: cpu
Connected successfully to MySQL database: drugbank
Distinct SMILES property keys found in DB: [{'kind': 'SMILES'}]
Extracted chemical records (D_chem): 600
Extracted drug-target records (D_val): 600
Sanitized single-molecule candidates: 0


[20:10:58] SMILES Parse Error: syntax error while parsing: smiles
[20:10:58] SMILES Parse Error: check for mistakes around position 2:
[20:10:58] smiles
[20:10:58] ~^
[20:10:58] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[20:10:58] SMILES Parse Error: syntax error while parsing: smiles
[20:10:58] SMILES Parse Error: check for mistakes around position 2:
[20:10:58] smiles
[20:10:58] ~^
[20:10:58] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[20:10:58] SMILES Parse Error: syntax error while parsing: smiles
[20:10:58] SMILES Parse Error: check for mistakes around position 2:
[20:10:58] smiles
[20:10:58] ~^
[20:10:58] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[20:10:58] SMILES Parse Error: syntax error while parsing: smiles
[20:10:58] SMILES Parse Error: check for mistakes around position 2:
[20:10:58] smiles
[20:10:58] ~^
[20:10:58] SMILES Parse Error: Failed parsing SMILES 'smiles' for input: 'smiles'
[20:

Constructing 3D Attributed Graphs: 0it [00:00, ?it/s]


Phase 1 Complete: 0 spatial graphs successfully constructed.


RuntimeError: [FATAL] Zero molecular graphs were created from the extracted DrugBank SMILES.

In [9]:
import pymysql
from pymysql.cursors import DictCursor

conn = pymysql.connect(
    host="127.0.0.1", port=3306, user="root", password="",
    database="drugbank", charset="utf8mb4", cursorclass=DictCursor
)
with conn.cursor() as cur:
    # Check what 'kind' values exist in drug_property
    cur.execute("""
        SELECT kind, COUNT(*) AS count 
        FROM drug_property 
        GROUP BY kind 
        ORDER BY count DESC 
        LIMIT 20;
    """)
    print("--- TOP 20 PROPERTY KINDS IN drug_property ---")
    for row in cur.fetchall():
        print(f"kind: {repr(row['kind'])}, count: {row['count']}")

    # Preview actual SMILES values
    cur.execute("""
        SELECT kind, value 
        FROM drug_property 
        WHERE LOWER(kind) LIKE '%smiles%' 
        LIMIT 5;
    """)
    print("\n--- SAMPLE SMILES ROWS ---")
    for row in cur.fetchall():
        print(f"kind: {repr(row['kind'])}, value: {repr(row['value'])}")
conn.close()

--- TOP 20 PROPERTY KINDS IN drug_property ---
kind: 'logP', count: 24503
kind: 'Water Solubility', count: 12706
kind: 'Molecular Weight', count: 12650
kind: 'Molecular Formula', count: 12575
kind: 'Monoisotopic Weight', count: 12316
kind: 'InChI', count: 12316
kind: 'InChIKey', count: 12316
kind: 'SMILES', count: 12313
kind: 'Rule of Five', count: 11929
kind: 'Ghose Filter', count: 11929
kind: 'Refractivity', count: 11922
kind: 'Polarizability', count: 11922
kind: 'Rotatable Bond Count', count: 11922
kind: 'H Bond Acceptor Count', count: 11922
kind: 'H Bond Donor Count', count: 11922
kind: 'Number of Rings', count: 11922
kind: 'Bioavailability', count: 11922
kind: 'Traditional IUPAC Name', count: 11922
kind: 'MDDR-Like Rule', count: 11922
kind: 'Polar Surface Area (PSA)', count: 11922

--- SAMPLE SMILES ROWS ---
kind: 'SMILES', value: 'CC[C@H](C)[C@H](NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CCC(O)=O)NC(=O)[C@H](CC1=CC=CC=C1)NC(=O)[C@H](CC(O)=O)NC(=O)CNC(=O)[C@H](CC(N)=O)NC(=O)CNC(=O)CNC(=O)C